# Loading NevIR as a real-world negation minimal-pair dataset

[NevIR](https://arxiv.org/abs/2305.07614) (Weller et al.) contains **contrastive query–document pairs**
built on CondaQA: two queries that differ *only* by negation, each matching exactly one of two minimally
different documents. Here we harvest the **queries**, not the corpus, for three reasons:

1. **CLIP truncates at 77 tokens.** NevIR passages run several hundred tokens, and the negation edit often
   sits past the cut — the displacement Δ would then be measured on text that no longer contains the
   negation. Queries are one sentence long.
2. Queries are genuine **minimal pairs** (`1000-2_q1` / `1000-2_q2`), which is exactly the input the rank
   analysis needs.
3. The project's IR angle is about **negation in the query**, so this is the operative object anyway.

### Two problems this notebook has to solve

**(a) Orientation is not given.** The `q1`/`q2` order does *not* consistently run affirmative → negated:

| pair | q1 | q2 |
|---|---|---|
| `1000-2` | "had **always** been unoccupied" | "had **not** always been unoccupied" |
| `1000-3` | "What **un**occupied islands…" | "What occupied islands…" |

The direction flips between the two. Since $\Delta = e(\text{neg}) - e(\text{aff})$, a wrong sign flips the
vector; the *spectrum* is sign-invariant but the **mean displacement — the whole H1 signal — is destroyed**
by mixed orientations. So we orient every pair explicitly and flag the ones we cannot.

**(b) No category labels.** NevIR is not annotated by negation mechanism, so we classify each pair
ourselves using the project taxonomy, exploiting the fact that these are minimal pairs: the *diff* between
the two sentences localises the negation edit far more reliably than a lexicon scan of either sentence.

In [ ]:
from __future__ import annotations

import difflib
import json
import re
import time
import urllib.parse
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd

API = "https://datasets-server.huggingface.co/rows"
DATASET = "mteb/NevIR"
DS_CONFIG = "queries"      # "corpus" holds the passages; queries are the minimal pairs we want
SPLIT = "test"
PAGE = 100                 # datasets-server caps `length` at 100
OUT_PARQUET = Path("../data/nevir_pairs.parquet")
MAX_ROWS = None            # None = fetch everything (2766 queries -> 1383 pairs)


def fetch_rows(dataset=DATASET, config=DS_CONFIG, split=SPLIT, page=PAGE, max_rows=MAX_ROWS, pause=0.15):
    """Page through the HF datasets-server REST API (same endpoint as the curl one-liner)."""
    rows, offset, total = [], 0, None
    while True:
        qs = urllib.parse.urlencode({"dataset": dataset, "config": config, "split": split,
                                     "offset": offset, "length": page})
        url = f"{API}?{qs}"
        for attempt in range(4):
            try:
                with urllib.request.urlopen(url, timeout=60) as r:
                    payload = json.loads(r.read())
                break
            except Exception as e:
                if attempt == 3:
                    raise
                time.sleep(1.5 * (attempt + 1))

        total = payload.get("num_rows_total", total)
        batch = [r["row"] for r in payload.get("rows", [])]
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        print(f"\r  fetched {len(rows)}/{total}", end="")
        if (max_rows and len(rows) >= max_rows) or (total and offset >= total):
            break
        time.sleep(pause)

    print()
    return rows[:max_rows] if max_rows else rows


raw = fetch_rows()
print(f"{len(raw)} query rows | example: {raw[0]}")

## 1. Pairing

Query ids follow `<passage>-<item>_q1` / `_q2`. The part before `_q` identifies the minimal pair, and the
part before the first `-` identifies the **source CondaQA passage** — several pairs share a passage, which
gives us a content-control variable for the `category × content` variance decomposition in notebook 01.

In [ ]:
by_key = {}
for r in raw:
    m = re.match(r"^(?P<key>.+)_q(?P<slot>[12])$", str(r["_id"]))
    if not m:
        continue
    by_key.setdefault(m.group("key"), {})[m.group("slot")] = r["text"]

pairs = [{"key": k, "passage": k.split("-")[0], "q1": v["1"], "q2": v["2"]}
         for k, v in by_key.items() if "1" in v and "2" in v]
print(f"{len(pairs)} complete minimal pairs from {len({p['passage'] for p in pairs})} source passages")
for p in pairs[:3]:
    print(f"\n  [{p['key']}]\n    q1: {p['q1']}\n    q2: {p['q2']}")

## 2. Taxonomy lexicon + minimal-pair classifier

Two complementary signals, applied in order:

1. **Cue lexicon** (particle, quantifier, adverb, prepositional, implicit-verb, exclusion, temporal
   cessation). The member carrying strictly more cues is the negated one — this handles multiword cues
   (`no longer`, `other than`) that a token diff would fragment.
2. **Affix diff.** For pairs the lexicon cannot separate, align the two token sequences and look for a
   word related to its counterpart by a negative affix (`occupied` → `unoccupied`, `possible` →
   `impossible`, `hope` → `hopeless`). This is *much* more reliable than scanning for prefixes in a single
   sentence, where `international`, `individual` or `district` would all be false positives: here the
   counterpart word has to actually be present in the sibling sentence.

`mis-` and `anti-`/`counter-` are captured too, but tagged `is_distractor=True`: they look morphologically
negative while not being semantic negation (`misunderstand` ≠ `not understand`), so they serve as the
validity control in notebook 01 §9.

In [ ]:
CUES = {
    "particle_not":   ["not", "cannot"],
    "quantifier_no":  ["no", "none", "nobody", "nothing", "no one", "noone", "nowhere", "neither", "nor"],
    "adverb_never":   ["never", "rarely", "hardly", "barely", "scarcely", "seldom"],
    "prep_without":   ["without", "lacking", "devoid of", "absent", "missing", "free of", "free from"],
    "implicit_verb":  ["refuse", "refused", "refuses", "deny", "denied", "denies", "reject", "rejected",
                       "fail", "failed", "fails", "avoid", "avoided", "exclude", "excluded", "excludes",
                       "omit", "omitted", "prevent", "prevented", "prevents", "forbid", "forbade",
                       "doubt", "doubted", "lack", "lacked", "lacks", "unable", "stopped", "ceased",
                       "removed", "lost", "denying", "failing"],
    "exclusion":      ["except", "besides", "other than", "apart from", "aside from", "excluding",
                       "save for", "but not"],
    "temporal_cessation": ["no longer", "not yet", "anymore", "any longer", "used to"],
}

NEG_PREFIXES = ["un", "in", "im", "il", "ir", "non", "dis", "a"]
DISTRACTOR_PREFIXES = ["mis", "anti", "counter"]
NEG_SUFFIXES = ["less", "free"]

CONTRACTIONS = {"n't": " not", "cannot": "can not", "can't": "can not", "won't": "will not",
                "shan't": "shall not", "ain't": "is not"}


def normalise(text: str) -> str:
    t = " " + text.lower().strip() + " "
    for k, v in CONTRACTIONS.items():
        t = t.replace(k, v)
    return re.sub(r"\s+", " ", t)


def tokens(text: str) -> list[str]:
    # Hyphenated words stay glued: "non-Czech" must remain one token, otherwise the affix test
    # compares "non" and "czech" separately and never sees the prefix relation.
    return re.findall(r"[a-z]+(?:-[a-z]+)*", normalise(text))


def cue_profile(text: str) -> Counter:
    t = normalise(text)
    prof = Counter()
    for cat, cues in CUES.items():
        for cue in cues:
            n = len(re.findall(rf"\b{re.escape(cue)}\b", t))
            if n:
                prof[cat] += n
    return prof


def affix_relation(w_a: str, w_b: str):
    """If w_a/w_b differ by a negative affix, return (negated_word, category, subclass, distractor)."""
    orig_a, orig_b = w_a, w_b
    w_a, w_b = w_a.replace("-", ""), w_b.replace("-", "")   # "non-czech" ~ "nonczech" vs "czech"
    back = {w_a: orig_a, w_b: orig_b}
    for base, other in ((w_a, w_b), (w_b, w_a)):
        for p in DISTRACTOR_PREFIXES:
            if other == p + base and len(base) >= 4:
                return back[other], "distractor_" + p, p, True
        for p in NEG_PREFIXES:
            if other == p + base and len(base) >= (5 if p == "a" else 3):
                return back[other], "affixal_prefix", p, False
        for s in NEG_SUFFIXES:
            if other in (base + s, base.rstrip("e") + s) and len(base) >= 3:
                return back[other], "affixal_suffix", s, False
    return None


def classify_pair(q1: str, q2: str) -> dict:
    """Decide which member is negated and by which mechanism."""
    p1, p2 = cue_profile(q1), cue_profile(q2)

    # ---- signal 1: cue lexicon ------------------------------------------------
    if sum(p1.values()) != sum(p2.values()):
        neg_is_q2 = sum(p2.values()) > sum(p1.values())
        rich, poor = (p2, p1) if neg_is_q2 else (p1, p2)
        deltas = {c: rich[c] - poor.get(c, 0) for c in rich if rich[c] - poor.get(c, 0) > 0}
        if deltas:
            cat = max(deltas, key=deltas.get)
            cue_hit = next((c for c in CUES[cat] if re.search(rf"\b{re.escape(c)}\b",
                                                              normalise(q2 if neg_is_q2 else q1))), cat)
            return {"category": cat, "subclass": cue_hit, "negated_is_q2": neg_is_q2,
                    "is_distractor": False, "oriented": True}

    # ---- signal 2: affix diff -------------------------------------------------
    t1, t2 = tokens(q1), tokens(q2)
    sm = difflib.SequenceMatcher(a=t1, b=t2, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        for w_a in t1[i1:i2] or [""]:
            for w_b in t2[j1:j2] or [""]:
                rel = affix_relation(w_a, w_b)
                if rel:
                    neg_word, cat, sub, dist = rel
                    return {"category": cat, "subclass": sub, "negated_is_q2": neg_word in t2,
                            "is_distractor": dist, "oriented": True}

    # ---- fallback: lexical substitution, direction unknown --------------------
    return {"category": "other_substitution", "subclass": "na", "negated_is_q2": True,
            "is_distractor": False, "oriented": False}


rows = []
for p in pairs:
    c = classify_pair(p["q1"], p["q2"])
    neg, aff = (p["q2"], p["q1"]) if c["negated_is_q2"] else (p["q1"], p["q2"])
    rows.append({
        "pair_id": p["key"], "object": p["passage"],       # `object` = source passage -> content control
        "affirmative": aff, "negative": neg,
        "category": c["category"], "subclass": c["subclass"],
        "scope": "sentential", "is_distractor": c["is_distractor"], "oriented": c["oriented"],
    })

df = pd.DataFrame(rows)
print(f"{len(df)} pairs | {df['oriented'].mean():.1%} confidently oriented")

## 3. What NevIR actually contains

The distribution below is itself a finding worth keeping: it shows how uneven NevIR's coverage of negation
mechanisms is — the same imbalance Petcu et al. report when they run their logic-based classifier over it.
Any rank estimated on this data inherits that imbalance, which is precisely why a purpose-built factorial
dataset is still needed afterwards.

In [ ]:
dist = (df.groupby(["category", "is_distractor"])
          .agg(n=("pair_id", "size"), oriented=("oriented", "mean"))
          .sort_values("n", ascending=False))
display(dist)

print("\nExamples per category:\n" + "=" * 100)
for cat in dist.reset_index()["category"]:
    sub = df[df["category"] == cat].head(2)
    print(f"\n[{cat}]  n={int(dist.loc[(cat, sub['is_distractor'].iloc[0]), 'n'])}")
    for _, r in sub.iterrows():
        print(f"   AFF: {r['affirmative']}")
        print(f"   NEG: {r['negative']}")

In [ ]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PARQUET, index=False)
print(f"wrote {len(df)} pairs -> {OUT_PARQUET.resolve()}")
print("\nColumns:", list(df.columns))
print("\nNext: set CONFIG['parquet_path'] = '../data/nevir_pairs.parquet' in 01_negation_subspace_rank.ipynb")

## Caveats before reading anything into the results

* **`other_substitution` is not a negation category.** Those pairs differ by an antonym or a rephrasing that
  the classifier could not localise, and their orientation is arbitrary. Notebook 01 should either drop them
  or treat them as a separate class — pooling them into the "negation" set inflates the measured rank with
  generic lexical variation, which is exactly the artefact we are trying to avoid.
* **Orientation errors flip Δ.** They leave the spectrum untouched (a sign flip does not change the
  second-moment matrix) but they attenuate the mean displacement, so the translation share ρ measured on
  NevIR is a **lower bound** on the true one.
* **Register is uniform.** Every NevIR query is an interrogative sentence, so any direction found here is
  entangled with question form; the control directions (plural / tense / question) from the project plan
  are what would disentangle that.
* **Not factorial.** Categories and source passages are not crossed by design, so the `category × object`
  decomposition is approximate here — it becomes exact only on the purpose-built dataset.